# Stage 0. 평가 기준 정하기

In [2]:
# %% [Cell 0] 설정 & 임포트 ---------------------------------------------------
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

DATA_PATH = r"C:\Users\rla_r\OneDrive\바탕 화면\학습데이터_최종_캐나다지수.csv"

TARGET = "Target"

# (1) 누수: 레이블을 그대로 흘리는 컬럼 → 절대 피처로 쓰지 않음
LEAKAGE_COLS = ["샘플유형", "시간샘플링방식", "공간층"]

# (2) 식별자/메타: 피처는 아니지만 점검·가중치·시점 용도로 df에는 남겨둠
ID_META_COLS = [
    "샘플ID",
    "기준시각",
    "기상셀ID",
    "월_key",
    "시간_key",
    "실험안",
    "샘플가중치",
]
COORD_COLS = ["위도", "경도"]  # 피처로는 제외하되, 공간 블록/근접 점검에 필요

# (3) 트리 모델 기준 중복 후보 (단조변환·결정적 변환) → 상관 확인 후 제거
REDUNDANT_COLS = [
    "log1p_도로_최단거리_m",
    "log1p_임도_최단거리_m",
    "log1p_산림지역_최단거리_m",
    "FFMC_논문식_발생확률",
    "시점_풍향_deg",
    "시점_해면기압_hPa",
]

BLOCK_KEYS = ["기상셀ID", "기후지형유형"]  # 공간 블록 후보 기준


def cols_in(df, names):
    return [c for c in names if c in df.columns]

In [3]:
# %% [Cell 1] 로드 & 기본 무결성 (필수) --------------------------------------
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print(f"shape: {df.shape}  (README 기대값: 17045 x 68)")
print("\n[결측치 합계가 0이 아닌 컬럼만 표시]")
na = df.isna().sum()
print(na[na > 0] if (na > 0).any() else "  결측 없음 (0건) ✔")

print("\n[자료형 분포]")
print(df.dtypes.value_counts())

print("\n[기후지형유형 범주 — 유일한 범주형 피처가 깨끗한지]")
if "기후지형유형" in df.columns:
    print(df["기후지형유형"].value_counts(dropna=False))

# 누락 컬럼 경고
for grp, names in [
    ("누수", LEAKAGE_COLS),
    ("좌표", COORD_COLS),
    ("블록키", BLOCK_KEYS),
]:
    missing = [c for c in names if c not in df.columns]
    if missing:
        print(f"⚠ {grp} 컬럼 누락: {missing}  (컬럼명을 README와 대조하세요)")

shape: (17045, 68)  (README 기대값: 17045 x 68)

[결측치 합계가 0이 아닌 컬럼만 표시]
  결측 없음 (0건) ✔

[자료형 분포]
float64    55
object      8
int64       5
Name: count, dtype: int64

[기후지형유형 범주 — 유일한 범주형 피처가 깨끗한지]
기후지형유형
영서 내륙형    8453
영동 해안형    6157
고지·산간형    2435
Name: count, dtype: int64


In [4]:
# %% [Cell 2] 발생률 & PR-AUC 기준선 (필수) ----------------------------------
n = len(df)
n_pos = int((df[TARGET] == 1).sum())
n_neg = int((df[TARGET] == 0).sum())
prevalence = n_pos / n

print(f"양성(Target=1): {n_pos}    음성(Target=0): {n_neg}    전체: {n}")
print(f"발생률(prevalence): {prevalence:.4f}  (양성:음성 ≈ 1:{n_neg / n_pos:.1f})")
print(f"\n무작위 모델 기준선  →  PR-AUC ≈ {prevalence:.4f} ,  ROC-AUC = 0.5")
print("※ PR-AUC는 이 발생률이 바닥이다. 0.09를 의미있게 넘는지로 해석할 것.")

양성(Target=1): 1553    음성(Target=0): 15492    전체: 17045
발생률(prevalence): 0.0911  (양성:음성 ≈ 1:10.0)

무작위 모델 기준선  →  PR-AUC ≈ 0.0911 ,  ROC-AUC = 0.5
※ PR-AUC는 이 발생률이 바닥이다. 0.09를 의미있게 넘는지로 해석할 것.


In [5]:
# %% [Cell 3] 누수 컬럼 확인 — Target과 완벽히 정렬되는가 (필수) --------------
for c in cols_in(df, LEAKAGE_COLS):
    ct = pd.crosstab(df[c], df[TARGET])
    # 각 범주가 한쪽 Target에만 속하면 완전 누수
    pure = (ct == 0).any(axis=1)
    print(f"\n[{c}] vs Target  (한 범주가 Target 한쪽에만 있으면 = 누수)")
    print(ct)
    print(f"  → 완전 분리(누수) 범주: {list(ct.index[pure])}")
print("\n결론: 위 컬럼들은 X에서 제외 (LEAKAGE_COLS).")


[샘플유형] vs Target  (한 범주가 Target 한쪽에만 있으면 = 누수)
Target         0     1
샘플유형                  
Target_0A   7738     0
Target_0B1  4652     0
Target_0B2  3102     0
Target_1       0  1553
  → 완전 분리(누수) 범주: ['Target_0A', 'Target_0B1', 'Target_0B2', 'Target_1']

[시간샘플링방식] vs Target  (한 범주가 Target 한쪽에만 있으면 = 누수)
Target       0     1
시간샘플링방식             
균등무작위     3102     0
기후지형유형분포  4652     0
실제발생         0  1553
월시간대매칭    7738     0
  → 완전 분리(누수) 범주: ['균등무작위', '기후지형유형분포', '실제발생', '월시간대매칭']

[공간층] vs Target  (한 범주가 Target 한쪽에만 있으면 = 누수)
Target      0     1
공간층                
산림 내부    1550     0
산림 접근권   2330     0
생활권-WUI  3874     0
실제발생위치   7738  1553
  → 완전 분리(누수) 범주: ['산림 내부', '산림 접근권', '생활권-WUI']

결론: 위 컬럼들은 X에서 제외 (LEAKAGE_COLS).


In [6]:
# %% [Cell 4] 블록별 양성 개수 — ★ 블록 설계를 결정하는 핵심 점검 ------------
# 공간 블록 CV에서 평가폴드에 양성이 0이면 PR-AUC/F1이 정의되지 않는다.
for key in cols_in(df, BLOCK_KEYS):
    g = df.groupby(key)[TARGET].agg(전체="size", 양성="sum")
    g["양성비율"] = (g["양성"] / g["전체"]).round(3)
    g = g.sort_values("양성")

    n_blocks = len(g)
    zero = int((g["양성"] == 0).sum())
    low = int(((g["양성"] >= 1) & (g["양성"] < 5)).sum())
    ok = int((g["양성"] >= 5).sum())

    print(f"\n================ 블록 기준: {key}  (블록 {n_blocks}개) ================")
    print(f"  양성 0개 블록: {zero}   |   1~4개: {low}   |   5개 이상: {ok}")
    print("  [양성 적은 하위 10개 블록]")
    print(g.head(10).to_string())

    if zero > 0 or low > n_blocks * 0.3:
        print(
            f"  ⚠ {key} 단위는 양성이 너무 희박 → 그대로 폴드로 쓰면 빈/불안정 폴드 발생"
        )
        print("    → 인접 블록을 5~8개 묶음으로 그룹핑하거나, 더 굵은 기준 사용 권장")
    else:
        print(f"  ✔ {key} 단위는 폴드로 사용 가능 수준")

# 참고: 셀을 묶을 때 쓸 셀 중심좌표 (인접 그룹핑/KMeans 입력용)
if set(["기상셀ID"] + COORD_COLS).issubset(df.columns):
    centroids = df.groupby("기상셀ID")[COORD_COLS].mean()
    print(f"\n[기상셀 중심좌표 — 인접 블록 그룹핑 입력] 셀 {len(centroids)}개")
    print(centroids.head())


================ 블록 기준: 기상셀ID  (블록 90개) ================
  양성 0개 블록: 3   |   1~4개: 15   |   5개 이상: 72
  [양성 적은 하위 10개 블록]
         전체  양성   양성비율
기상셀ID                 
YS_0067  36   0  0.000
YS_0046  37   0  0.000
YS_0048  17   0  0.000
YS_0047  28   1  0.036
YS_0036  59   1  0.017
YS_0011  41   1  0.024
YS_0053  33   1  0.030
YS_0031  77   1  0.013
YD_0010  90   1  0.011
YS_0063   8   1  0.125
  ⚠ 기상셀ID 단위는 양성이 너무 희박 → 그대로 폴드로 쓰면 빈/불안정 폴드 발생
    → 인접 블록을 5~8개 묶음으로 그룹핑하거나, 더 굵은 기준 사용 권장

================ 블록 기준: 기후지형유형  (블록 3개) ================
  양성 0개 블록: 0   |   1~4개: 0   |   5개 이상: 3
  [양성 적은 하위 10개 블록]
          전체   양성   양성비율
기후지형유형                  
고지·산간형  2435  222  0.091
영동 해안형  6157  561  0.091
영서 내륙형  8453  770  0.091
  ✔ 기후지형유형 단위는 폴드로 사용 가능 수준

[기상셀 중심좌표 — 인접 블록 그룹핑 입력] 셀 90개
                위도          경도
기상셀ID                         
YD_0001  37.729251  128.895013
YD_0002  38.267824  128.514728
YD_0003  37.513831  129.102345
YD_0004  38.131282  128.592016
YD_0005  38.35

## 추가분석

In [7]:
# %% [Cell H0] 임포트 & 설정 -------------------------------------------------
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_recall_curve,
)

ZONE_COL = "기후지형유형"
CELL_COL = "기상셀ID"
COORD = ("위도", "경도")
N_PER_ZONE = {"영동 해안형": 2, "영서 내륙형": 2, "고지·산간형": 1}
SEED = 42  # 폴드 재현성 고정 — 전 Stage에서 동일 시드 사용


# %% [Cell H1] 존 인식 공간 블록 구성 ---------------------------------------
def build_spatial_blocks(
    df,
    zone_col=ZONE_COL,
    cell_col=CELL_COL,
    coord=COORD,
    n_per_zone=N_PER_ZONE,
    seed=SEED,
):
    cell = df.groupby(cell_col).agg(
        lat=(coord[0], "mean"),
        lon=(coord[1], "mean"),
        zone=(zone_col, lambda s: s.value_counts().index[0]),
    )
    cell["block"] = None
    for z, sub in cell.groupby("zone"):
        k = n_per_zone.get(z, 1) if isinstance(n_per_zone, dict) else n_per_zone
        k = min(k, len(sub))
        if k <= 1:
            labels = np.zeros(len(sub), dtype=int)
        else:
            km = KMeans(n_clusters=k, random_state=seed, n_init=10)
            labels = km.fit_predict(sub[["lat", "lon"]].to_numpy())
        cell.loc[sub.index, "block"] = [f"{z}#{l}" for l in labels]
    return df[cell_col].map(cell["block"]), cell


block, cell_table = build_spatial_blocks(df)
df["_block"] = block.values

bsum = df.groupby("_block")[TARGET].agg(전체="size", 양성="sum")
bsum["양성비율"] = (bsum["양성"] / bsum["전체"]).round(3)
print(f"총 블록 수: {df['_block'].nunique()}   (목표: 각 블록 양성 수가 고르게)")
print(bsum.sort_values("양성").to_string())
print("\n※ 특정 블록 양성이 과도하게 적/많으면 N_PER_ZONE를 조정해 재실행.")


# %% [Cell H2] 최종 테스트 블록 격리 ----------------------------------------
def select_test_block(df, bsum, target=None):
    """대표성 기준: 양성비율이 전체와 가장 가까운 블록을 최종 테스트로."""
    overall = (df["_block"].notna()).pipe(lambda _: (df[TARGET] == 1).mean())
    rep = (bsum["양성비율"] - overall).abs().sort_values()
    return rep.index[0]


TEST_BLOCK = "영서 내륙형#1"  # 자동선택 대신 수동 지정 (세 존을 CV에 모두 남김)
test_mask = df["_block"] == TEST_BLOCK
cv_blocks = [b for b in df["_block"].unique() if b != TEST_BLOCK]

print(f"격리할 최종 테스트 블록: {TEST_BLOCK}")
print(
    f"  테스트: 행 {int(test_mask.sum())}, 양성 {int(df.loc[test_mask, TARGET].sum())}"
)
print(f"  Stage 1~4 CV 폴드로 사용할 블록 {len(cv_blocks)}개: {cv_blocks}")
print("  ★ 이 테스트 블록은 Stage 5 최종 1회 외에는 절대 열지 않는다.")


# %% [Cell H3] evaluate() 지표 패널 -----------------------------------------
def evaluate(y_true, y_prob, threshold=0.5):
    """모델 순위는 PR_AUC로. F1/정밀도/재현율은 threshold 의존(진단)."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    out = {
        "PR_AUC": average_precision_score(y_true, y_prob),  # 주 지표
        "ROC_AUC": (
            roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
        ),
        "Brier": brier_score_loss(y_true, y_prob),  # 보정
    }
    y_pred = (y_prob >= threshold).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    out["Precision"] = prec
    out["Recall"] = rec
    out["F1"] = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    out["threshold"] = threshold
    return out


def pick_threshold(y_true, y_prob):
    """F1 최대 임계값. 검증셋에서만 호출 — 테스트셋엔 절대 쓰지 말 것."""
    p, r, th = precision_recall_curve(y_true, y_prob)
    if len(th) == 0:
        return 0.5
    f1 = np.divide(
        2 * p[:-1] * r[:-1],
        p[:-1] + r[:-1],
        out=np.zeros(len(th)),
        where=(p[:-1] + r[:-1]) > 0,
    )
    return float(th[int(np.argmax(f1))])


def summarize_cv(results):
    """폴드별 결과 리스트 → 지표별 평균±표준편차 (안정성 점검)."""
    keys = [k for k in results[0] if k != "threshold"]
    rows = {
        k: [np.mean([r[k] for r in results]), np.std([r[k] for r in results])]
        for k in keys
    }
    return pd.DataFrame(rows, index=["mean", "std"]).T.round(4)


# 동작 확인: 무작위 확률로 evaluate가 PR-AUC 바닥선(≈0.091)을 내는지
_y = df[TARGET].to_numpy()
_rng = np.random.default_rng(SEED)
print("[sanity] 무작위 예측 evaluate (PR_AUC가 발생률 ≈ 0.091 근처여야 정상)")
print(pd.Series(evaluate(_y, _rng.random(len(_y)))).round(4).to_string())

c:\Users\rla_r\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


총 블록 수: 5   (목표: 각 블록 양성 수가 고르게)
            전체   양성   양성비율
_block                    
영동 해안형#0  2181  220  0.101
고지·산간형#0  2435  222  0.091
영서 내륙형#0  3569  331  0.093
영동 해안형#1  3976  341  0.086
영서 내륙형#1  4884  439  0.090

※ 특정 블록 양성이 과도하게 적/많으면 N_PER_ZONE를 조정해 재실행.
격리할 최종 테스트 블록: 영서 내륙형#1
  테스트: 행 4884, 양성 439
  Stage 1~4 CV 폴드로 사용할 블록 4개: ['영동 해안형#1', '영서 내륙형#0', '영동 해안형#0', '고지·산간형#0']
  ★ 이 테스트 블록은 Stage 5 최종 1회 외에는 절대 열지 않는다.
[sanity] 무작위 예측 evaluate (PR_AUC가 발생률 ≈ 0.091 근처여야 정상)
PR_AUC       0.0967
ROC_AUC      0.5132
Brier        0.3303
Precision    0.0948
Recall       0.5216
F1           0.1604
threshold    0.5000


c:\Users\rla_r\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


# Stage 1. 피처셋 스크리닝

In [8]:
# %% [Cell S0] 제외 컬럼 정의 & 피처 그룹 자동 구성 ---------------------------
import numpy as np
import pandas as pd

# Stage 0에서 확정한 제외 컬럼 (자체 재선언 — 하네스 상태에 의존하지 않게)
TARGET = "Target"
LEAKAGE_COLS = ["샘플유형", "시간샘플링방식", "공간층"]
ID_META_COLS = [
    "샘플ID",
    "기준시각",
    "기상셀ID",
    "월_key",
    "시간_key",
    "실험안",
    "샘플가중치",
]
COORD_COLS = ["위도", "경도"]
REDUNDANT_COLS = [
    "log1p_도로_최단거리_m",
    "log1p_임도_최단거리_m",
    "log1p_산림지역_최단거리_m",
    "FFMC_논문식_발생확률",
    "시점_풍향_deg",
    "시점_해면기압_hPa",
]

DROP = set(
    LEAKAGE_COLS + ID_META_COLS + COORD_COLS + REDUNDANT_COLS + [TARGET, "_block"]
)
candidate = [c for c in df.columns if c not in DROP]

# 패턴으로 역할 그룹 분류
terrain = [
    c for c in candidate if any(k in c for k in ["고도", "경사", "사면방향", "TPI"])
]
distance = [c for c in candidate if "최단거리" in c]
canada = [
    c
    for c in candidate
    if any(k in c for k in ["FFMC", "DMC", "DC", "ISI", "BUI", "FWI", "Indexed"])
]
fa = [c for c in candidate if c.endswith("_score")]
zone_raw = [c for c in candidate if c == "기후지형유형"]
weather = [
    c for c in candidate if c not in set(terrain + distance + canada + fa + zone_raw)
]

# 범주형 기후지형유형 → 원-핫 (트리 입력용). feat = df + 더미
zone_cols = []
feat = df.copy()
if zone_raw:
    d = pd.get_dummies(df["기후지형유형"], prefix="zone").astype(int)
    feat = pd.concat([feat, d], axis=1)
    zone_cols = list(d.columns)

print("=== 피처 그룹 확인 (이름이 의도대로 분류됐는지 눈으로 검증) ===")
for name, g in [
    ("지형", terrain),
    ("거리", distance),
    ("기후지형유형(더미)", zone_cols),
    ("원시기상", weather),
    ("FA", fa),
    ("캐나다", canada),
]:
    print(f"\n[{name}] ({len(g)}개)\n  {g}")

=== 피처 그룹 확인 (이름이 의도대로 분류됐는지 눈으로 검증) ===

[지형] (5개)
  ['고도(m)', '경사도(도)', '사면방향_sin', '사면방향_cos', 'TPI(지형위치지수)']

[거리] (6개)
  ['도로_최단거리_m', '시가화_최단거리_m', '농업_최단거리_m', '임도_최단거리_m', '등산로_최단거리_m', '산림_최단거리_m']

[기후지형유형(더미)] (3개)
  ['zone_고지·산간형', 'zone_영동 해안형', 'zone_영서 내륙형']

[원시기상] (24개)
  ['시점_기온_C', '시점_풍속_m_s', '시점_습도_pct', '직전24h_평균풍속', '직전24h_최대풍속', '직전48h_평균풍속', '직전48h_최대풍속', '직전24h_평균기온_C', '직전24h_평균습도', '직전24h_최소습도', '직전48h_평균습도', '직전48h_최소습도', '직전24h_강수량합', '직전48h_강수량합', '풍향_sin', '풍향_cos', '서풍계열_여부', '시점_현지기압_hPa', '기압변동_3h', 'D-1_최소습도_pct', 'D-1_평균습도_pct', 'D-1_강수량합_mm', 'D-2_최소습도_pct', 'D-3_최소습도_pct']

[FA] (5개)
  ['F1_score', 'F2_score', 'F3_score', 'F4_score', 'F5_score']

[캐나다] (8개)
  ['FFMC', 'FFMC_10일평균', 'Indexed_FFMC', 'DMC', 'DC', 'ISI', 'BUI', 'FWI']


In [9]:
# %% [Cell S1] 피처셋 S0~S5 정의 -------------------------------------------
static = terrain + distance + zone_cols
feature_sets = {
    "S0_정적": static,
    "S1_정적+원시기상": static + weather,
    "S2_정적+FA": static + fa,
    "S3_정적+캐나다": static + canada,
    "S4_정적+원시+캐나다": static + weather + canada,
    "S5_전부": static + weather + fa + canada,
}
print("=== 피처셋별 변수 개수 ===")
for k, v in feature_sets.items():
    print(f"  {k:20s}: {len(v)}개")

# %% [Cell S2] 모델 & 공간블록 CV 러너 -------------------------------------
from sklearn.ensemble import RandomForestClassifier

MODELS = {
    "RF": lambda: RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED)
}
try:
    from lightgbm import LGBMClassifier

    MODELS["LGBM"] = lambda: LGBMClassifier(
        n_estimators=400, n_jobs=-1, random_state=SEED, verbose=-1
    )
except ImportError:
    print("⚠ lightgbm 미설치 → 'pip install lightgbm' 후 재실행. 지금은 RF만 진행.")


def run_cv(feature_cols, model_factory):
    """테스트 블록 제외, cv_blocks를 leave-one-block-out으로 평가."""
    res = []
    for vb in cv_blocks:
        tr_idx = df.index[df["_block"].isin([b for b in cv_blocks if b != vb])]
        va_idx = df.index[df["_block"] == vb]
        Xtr, ytr = feat.loc[tr_idx, feature_cols], df.loc[tr_idx, TARGET]
        Xva, yva = feat.loc[va_idx, feature_cols], df.loc[va_idx, TARGET]
        m = model_factory()
        m.fit(Xtr, ytr)
        p = m.predict_proba(Xva)[:, 1]
        res.append(evaluate(yva.to_numpy(), p))
    return res

=== 피처셋별 변수 개수 ===
  S0_정적               : 14개
  S1_정적+원시기상          : 38개
  S2_정적+FA            : 19개
  S3_정적+캐나다           : 22개
  S4_정적+원시+캐나다        : 46개
  S5_전부               : 51개


In [10]:
pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


In [11]:
# %% [Cell S3] 전체 비교 실행 (★ 다소 무거움: 최대 6×2×4 = 48 fit) ----------
rows = []
for fs_name, cols in feature_sets.items():
    for mdl_name, factory in MODELS.items():
        s = summarize_cv(run_cv(cols, factory))  # index=지표, cols=[mean,std]
        rows.append(
            {
                "피처셋": fs_name,
                "모델": mdl_name,
                "n_feat": len(cols),
                "PR_AUC": s.loc["PR_AUC", "mean"],
                "PR_std": s.loc["PR_AUC", "std"],
                "ROC_AUC": s.loc["ROC_AUC", "mean"],
                "F1": s.loc["F1", "mean"],
                "Recall": s.loc["Recall", "mean"],
                "Brier": s.loc["Brier", "mean"],
            }
        )
        print(
            f"  done: {fs_name:20s} × {mdl_name:5s}  "
            f"PR-AUC={s.loc['PR_AUC','mean']:.4f} ± {s.loc['PR_AUC','std']:.4f}"
        )

table = pd.DataFrame(rows).sort_values("PR_AUC", ascending=False).reset_index(drop=True)
print("\n=== Stage 1 (선택지 A) 피처셋 × 모델 비교 — PR-AUC 내림차순 ===")
print(f"(무작위 바닥선 PR-AUC ≈ 0.091)\n")
print(table.round(4).to_string(index=False))

  done: S0_정적                × RF     PR-AUC=0.1514 ± 0.0036
  done: S0_정적                × LGBM   PR-AUC=0.1494 ± 0.0020
  done: S1_정적+원시기상           × RF     PR-AUC=0.2350 ± 0.0539
  done: S1_정적+원시기상           × LGBM   PR-AUC=0.2071 ± 0.0462
  done: S2_정적+FA             × RF     PR-AUC=0.1854 ± 0.0091
  done: S2_정적+FA             × LGBM   PR-AUC=0.1599 ± 0.0131
  done: S3_정적+캐나다            × RF     PR-AUC=0.3177 ± 0.1287
  done: S3_정적+캐나다            × LGBM   PR-AUC=0.2829 ± 0.1118
  done: S4_정적+원시+캐나다         × RF     PR-AUC=0.3304 ± 0.1362
  done: S4_정적+원시+캐나다         × LGBM   PR-AUC=0.2873 ± 0.1196
  done: S5_전부                × RF     PR-AUC=0.3046 ± 0.1079
  done: S5_전부                × LGBM   PR-AUC=0.2948 ± 0.1185

=== Stage 1 (선택지 A) 피처셋 × 모델 비교 — PR-AUC 내림차순 ===
(무작위 바닥선 PR-AUC ≈ 0.091)

         피처셋   모델  n_feat  PR_AUC  PR_std  ROC_AUC     F1  Recall  Brier
S4_정적+원시+캐나다   RF      46  0.3304  0.1362   0.8032 0.0072  0.0037 0.0733
   S3_정적+캐나다   RF      22  0.3177  0.1287   0

In [12]:
# %% 폴드별 PR-AUC 분해 (분산의 출처 확인)
for fs in ["S3_정적+캐나다", "S4_정적+원시+캐나다"]:
    print(f"\n[{fs}] × RF — 검증 블록별 PR-AUC")
    res = run_cv(feature_sets[fs], MODELS["RF"])
    for vb, r in zip(cv_blocks, res):
        npos = int((df.loc[df["_block"] == vb, TARGET] == 1).sum())
        print(
            f"  {vb:16s}: PR-AUC={r['PR_AUC']:.4f}  ROC={r['ROC_AUC']:.4f}  (양성 {npos})"
        )


[S3_정적+캐나다] × RF — 검증 블록별 PR-AUC
  영동 해안형#1        : PR-AUC=0.2797  ROC=0.8093  (양성 341)
  영서 내륙형#0        : PR-AUC=0.1954  ROC=0.7210  (양성 331)
  영동 해안형#0        : PR-AUC=0.5337  ROC=0.8521  (양성 220)
  고지·산간형#0        : PR-AUC=0.2618  ROC=0.7965  (양성 222)

[S4_정적+원시+캐나다] × RF — 검증 블록별 PR-AUC
  영동 해안형#1        : PR-AUC=0.3282  ROC=0.8239  (양성 341)
  영서 내륙형#0        : PR-AUC=0.1972  ROC=0.7365  (양성 331)
  영동 해안형#0        : PR-AUC=0.5517  ROC=0.8599  (양성 220)
  고지·산간형#0        : PR-AUC=0.2443  ROC=0.7924  (양성 222)


## CV

In [13]:
# %% [Cell Z0] 존 층화 공간 폴드 구성 --------------------------------------
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

ZONE_COL, CELL_COL, COORD = "기후지형유형", "기상셀ID", ("위도", "경도")
K_FOLDS = 4  # CV 폴드 수 (그룹 수 = K_FOLDS+1, 그룹0=테스트)
SEED = 42


def build_zone_stratified_folds(
    df, zone_col=ZONE_COL, cell_col=CELL_COL, coord=COORD, k_folds=K_FOLDS, seed=SEED
):
    n_groups = k_folds + 1
    cell = df.groupby(cell_col).agg(
        lat=(coord[0], "mean"),
        lon=(coord[1], "mean"),
        zone=(zone_col, lambda s: s.value_counts().index[0]),
    )
    cell["grp"] = -1
    for z, sub in cell.groupby("zone"):
        g = min(n_groups, len(sub))
        if g < n_groups:
            print(
                f"⚠ 존 '{z}' 셀 {len(sub)}개 < 그룹 {n_groups} → 일부 폴드에서 이 존이 빠질 수 있음"
            )
        if g <= 1:
            labels = np.zeros(len(sub), dtype=int)
        else:
            labels = KMeans(n_clusters=g, random_state=seed, n_init=10).fit_predict(
                sub[["lat", "lon"]].to_numpy()
            )
        cell.loc[sub.index, "grp"] = labels
    return df[cell_col].map(cell["grp"]).astype(int), cell


fold, cell_tbl = build_zone_stratified_folds(df)
df["_fold"] = fold.values
TEST_FOLD = 0
CV_FOLDS = [f for f in sorted(df["_fold"].unique()) if f != TEST_FOLD]

# 검증 1: 폴드 × 존 양성 수 — 모든 폴드(0 포함)가 세 존을 담는지 확인
pos = df[df[TARGET] == 1]
print("=== 폴드 × 존 : 양성 수  (fold 0 = 테스트) ===")
print(pd.crosstab(pos["_fold"], pos[ZONE_COL]).to_string())

# 검증 2: 폴드별 규모
gsz = df.groupby("_fold")[TARGET].agg(전체="size", 양성="sum")
gsz["양성비율"] = (gsz["양성"] / gsz["전체"]).round(3)
print("\n=== 폴드별 규모 ===")
print(gsz.to_string())
print(f"\nTEST_FOLD = 0 (격리),  CV_FOLDS = {CV_FOLDS}")
print("★ fold 0(테스트)은 Stage 5 최종 1회 외 절대 열지 않음")

=== 폴드 × 존 : 양성 수  (fold 0 = 테스트) ===
기후지형유형  고지·산간형  영동 해안형  영서 내륙형
_fold                         
0          103      39     181
1           10     132      78
2           60      51     121
3           40     125     263
4            9     214     127

=== 폴드별 규모 ===
         전체   양성   양성비율
_fold                  
0      4105  323  0.079
1      2519  220  0.087
2      1933  232  0.120
3      4632  428  0.092
4      3856  350  0.091

TEST_FOLD = 0 (격리),  CV_FOLDS = [1, 2, 3, 4]
★ fold 0(테스트)은 Stage 5 최종 1회 외 절대 열지 않음


c:\Users\rla_r\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\rla_r\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\rla_r\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [14]:
# %% [Cell Z1] 새 CV용 run_cv (이전 _block 기반 run_cv를 대체) --------------
def run_cv(feature_cols, model_factory, cv_folds=None):
    cv_folds = cv_folds or CV_FOLDS
    res = []
    for vf in cv_folds:
        tr = df.index[df["_fold"].isin([f for f in cv_folds if f != vf])]
        va = df.index[df["_fold"] == vf]
        m = model_factory()
        m.fit(feat.loc[tr, feature_cols], df.loc[tr, TARGET])
        p = m.predict_proba(feat.loc[va, feature_cols])[:, 1]
        res.append(evaluate(df.loc[va, TARGET].to_numpy(), p))
    return res


# %% [Cell Z2] 검증: S3 × RF를 새 CV로 재평가 (분산이 줄었는지) --------------
res = run_cv(feature_sets["S3_정적+캐나다"], MODELS["RF"])
print("[S3 × RF] 존 층화 CV — 폴드별 PR-AUC")
for f, r in zip(CV_FOLDS, res):
    print(f"  fold {f}: PR-AUC={r['PR_AUC']:.4f}  ROC={r['ROC_AUC']:.4f}")
print("\n요약 (평균 ± 표준편차):")
print(summarize_cv(res).to_string())
print("\n※ 이전 leave-one-region-out: PR-AUC 0.318 ± 0.129 였음.")
print("  std가 눈에 띄게 줄고 평균이 보간 수준(↑)으로 오르면 잣대 교정 성공.")

[S3 × RF] 존 층화 CV — 폴드별 PR-AUC
  fold 1: PR-AUC=0.4460  ROC=0.8360
  fold 2: PR-AUC=0.2566  ROC=0.7164
  fold 3: PR-AUC=0.2985  ROC=0.7529
  fold 4: PR-AUC=0.2369  ROC=0.7972

요약 (평균 ± 표준편차):
             mean     std
PR_AUC     0.3095  0.0819
ROC_AUC    0.7756  0.0451
Brier      0.0785  0.0125
Precision  0.5625  0.3698
Recall     0.0187  0.0286
F1         0.0346  0.0523

※ 이전 leave-one-region-out: PR-AUC 0.318 ± 0.129 였음.
  std가 눈에 띄게 줄고 평균이 보간 수준(↑)으로 오르면 잣대 교정 성공.


# Stage 2. 모델군 비교

In [18]:
# %% [Cell M0] 모델군 정의 (미설치 라이브러리는 건너뜀) ----------------------
from sklearn.ensemble import RandomForestClassifier

MODELS2 = {
    "RF": lambda: RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED)
}

try:
    from xgboost import XGBClassifier

    MODELS2["XGBoost"] = lambda: XGBClassifier(
        n_estimators=400,
        tree_method="hist",
        n_jobs=-1,
        random_state=SEED,
        eval_metric="logloss",
        verbosity=0,
    )
except ImportError:
    print("⚠ xgboost 미설치 → pip install xgboost")

try:
    from lightgbm import LGBMClassifier

    MODELS2["LightGBM"] = lambda: LGBMClassifier(
        n_estimators=400, n_jobs=-1, random_state=SEED, verbose=-1
    )
except ImportError:
    print("⚠ lightgbm 미설치 → pip install lightgbm")

try:
    from catboost import CatBoostClassifier

    MODELS2["CatBoost"] = lambda: CatBoostClassifier(
        iterations=400, random_seed=SEED, verbose=0
    )
except ImportError:
    print("⚠ catboost 미설치 → pip install catboost")

print("비교 대상 모델:", list(MODELS2))

비교 대상 모델: ['RF', 'XGBoost', 'LightGBM', 'CatBoost']


In [19]:
# %% [Cell M1] S3로 모델군 비교 실행 ---------------------------------------
STAGE2_FS = "S3_정적+캐나다"
cols = feature_sets[STAGE2_FS]

rows, perfold = [], {}
for name, factory in MODELS2.items():
    res = run_cv(cols, factory)  # 존 층화 CV
    s = summarize_cv(res)
    perfold[name] = [r["PR_AUC"] for r in res]
    rows.append(
        {
            "모델": name,
            "n_feat": len(cols),
            "PR_AUC": s.loc["PR_AUC", "mean"],
            "PR_std": s.loc["PR_AUC", "std"],
            "PR_보수적": s.loc["PR_AUC", "mean"]
            - s.loc["PR_AUC", "std"],  # 평균-표준편차
            "ROC_AUC": s.loc["ROC_AUC", "mean"],
            "Brier": s.loc["Brier", "mean"],
        }
    )
    print(
        f"  done {name:9s}: PR-AUC={s.loc['PR_AUC','mean']:.4f} ± {s.loc['PR_AUC','std']:.4f}"
    )

table = pd.DataFrame(rows).sort_values("PR_AUC", ascending=False).reset_index(drop=True)
print(f"\n=== Stage 2 모델군 비교 ({STAGE2_FS}) — PR-AUC 내림차순 ===")
print("(바닥선 PR-AUC ≈ 0.091)\n")
print(table.round(4).to_string(index=False))

# 모델 × 폴드 PR-AUC 행렬 — 지역 일관성 확인 (고르게 잘하는 모델 찾기)
pf = pd.DataFrame(perfold, index=[f"fold{f}" for f in CV_FOLDS]).T
print("\n=== 모델 × 폴드 PR-AUC (값이 폴드 간 고를수록 안정적) ===")
print(pf.round(3).to_string())

print("\n선택 가이드: PR_AUC 평균이 높으면서 PR_std 작은(=PR_보수적 큰) 모델을 우대.")
print("상위 2~3개를 Stage 3(Optuna 튜닝)로 올린다.")

  done RF       : PR-AUC=0.3095 ± 0.0819
  done XGBoost  : PR-AUC=0.2408 ± 0.0353
  done LightGBM : PR-AUC=0.2613 ± 0.0458
  done CatBoost : PR-AUC=0.2891 ± 0.0673

=== Stage 2 모델군 비교 (S3_정적+캐나다) — PR-AUC 내림차순 ===
(바닥선 PR-AUC ≈ 0.091)

      모델  n_feat  PR_AUC  PR_std  PR_보수적  ROC_AUC  Brier
      RF      22  0.3095  0.0819  0.2276   0.7756 0.0785
CatBoost      22  0.2891  0.0673  0.2218   0.7635 0.0805
LightGBM      22  0.2613  0.0458  0.2155   0.7544 0.0881
 XGBoost      22  0.2408  0.0353  0.2055   0.7348 0.0913

=== 모델 × 폴드 PR-AUC (값이 폴드 간 고를수록 안정적) ===
          fold1  fold2  fold3  fold4
RF        0.446  0.257  0.299  0.237
XGBoost   0.267  0.252  0.264  0.180
LightGBM  0.323  0.249  0.276  0.197
CatBoost  0.393  0.238  0.304  0.221

선택 가이드: PR_AUC 평균이 높으면서 PR_std 작은(=PR_보수적 큰) 모델을 우대.
상위 2~3개를 Stage 3(Optuna 튜닝)로 올린다.


# Stage 3. Optuna 튜닝

In [21]:
# %% [Cell T0] 탐색공간·빌더 정의 ------------------------------------------
import numpy as np
import pandas as pd
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    raise SystemExit("optuna 미설치 → pip install optuna 후 재실행")

N_TRIALS = 40
STAGE3_FS = "S3_정적+캐나다"
COLS = feature_sets[STAGE3_FS]


def make_factory(C, p, f):
    return lambda: C(**p, **f)


# --- 모델별 탐색공간 (trial → 모델 kwargs dict) ---
def suggest_rf(t):
    cw = t.suggest_categorical("class_weight", ["none", "balanced_subsample"])
    return dict(
        n_estimators=t.suggest_int("n_estimators", 200, 600),
        max_depth=t.suggest_int("max_depth", 4, 24),
        max_features=t.suggest_float("max_features", 0.2, 0.8),
        min_samples_leaf=t.suggest_int("min_samples_leaf", 1, 20),
        min_samples_split=t.suggest_int("min_samples_split", 2, 20),
        class_weight=None if cw == "none" else cw,
    )

def suggest_xgb(t):
    return dict(
        n_estimators=t.suggest_int("n_estimators", 200, 800),
        max_depth=t.suggest_int("max_depth", 3, 10),
        learning_rate=t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=t.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=t.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_weight=t.suggest_int("min_child_weight", 1, 10),
        reg_lambda=t.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        reg_alpha=t.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        gamma=t.suggest_float("gamma", 0.0, 5.0),
        scale_pos_weight=t.suggest_float("scale_pos_weight", 1.0, 15.0),
    )

def suggest_lgbm(t):
    return dict(
        n_estimators=t.suggest_int("n_estimators", 200, 800),
        learning_rate=t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        num_leaves=t.suggest_int("num_leaves", 15, 255),
        max_depth=t.suggest_int("max_depth", 3, 12),
        min_child_samples=t.suggest_int("min_child_samples", 5, 100),
        subsample=t.suggest_float("subsample", 0.6, 1.0),
        subsample_freq=t.suggest_int("subsample_freq", 1, 7),
        colsample_bytree=t.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_lambda=t.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        reg_alpha=t.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        scale_pos_weight=t.suggest_float("scale_pos_weight", 1.0, 15.0),
    )

def suggest_cat(t):
    aw = t.suggest_categorical("auto_class_weights", ["none", "Balanced"])
    return dict(
        iterations=t.suggest_int("iterations", 200, 800),
        learning_rate=t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        depth=t.suggest_int("depth", 4, 10),
        l2_leaf_reg=t.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        random_strength=t.suggest_float("random_strength", 0.0, 10.0),
        bagging_temperature=t.suggest_float("bagging_temperature", 0.0, 1.0),
        auto_class_weights=None if aw == "none" else aw,
    )


# --- 설치된 모델만 등록 ---
from sklearn.ensemble import RandomForestClassifier
SUGGEST, BUILDERS = {}, {}
SUGGEST["RF"] = suggest_rf
BUILDERS["RF"] = (RandomForestClassifier, dict(n_jobs=-1, random_state=SEED))
try:
    from xgboost import XGBClassifier
    SUGGEST["XGBoost"] = suggest_xgb
    BUILDERS["XGBoost"] = (XGBClassifier, dict(
        tree_method="hist", n_jobs=-1, random_state=SEED,
        eval_metric="logloss", verbosity=0))
except ImportError:
    print("⚠ xgboost 미설치 → 건너뜀")
try:
    from lightgbm import LGBMClassifier
    SUGGEST["LightGBM"] = suggest_lgbm
    BUILDERS["LightGBM"] = (LGBMClassifier, dict(n_jobs=-1, random_state=SEED, verbose=-1))
except ImportError:
    print("⚠ lightgbm 미설치 → 건너뜀")
try:
    from catboost import CatBoostClassifier
    SUGGEST["CatBoost"] = suggest_cat
    BUILDERS["CatBoost"] = (CatBoostClassifier, dict(random_seed=SEED, verbose=0))
except ImportError:
    print("⚠ catboost 미설치 → 건너뜀")


def objective(trial, name):
    params = SUGGEST[name](trial)
    Cls, fixed = BUILDERS[name]
    res = run_cv(COLS, make_factory(Cls, params, fixed))   # 존 층화 CV (테스트 폴드 미사용)
    return float(np.mean([r["PR_AUC"] for r in res]))

print("튜닝 대상:", list(SUGGEST), f"| trials/모델 = {N_TRIALS}")

튜닝 대상: ['RF', 'XGBoost', 'LightGBM', 'CatBoost'] | trials/모델 = 40


In [22]:
# %% [Cell T1] 튜닝 실행 (★ 가장 무거운 셀) --------------------------------
studies = {}
for name in SUGGEST:
    print(f"\n=== {name} 튜닝 ({N_TRIALS} trials) ===")
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda t, n=name: objective(t, n), n_trials=N_TRIALS)
    studies[name] = study
    print(f"  best mean PR-AUC = {study.best_value:.4f}")

# %% [Cell T2] 튜닝 결과 요약 & 기본값 대비 향상 ---------------------------
DEFAULT_PR = {"RF": 0.3095, "CatBoost": 0.2891, "LightGBM": 0.2613, "XGBoost": 0.2408}  # Stage 2

rows, TUNED = [], {}
for name, study in studies.items():
    bp = SUGGEST[name](optuna.trial.FixedTrial(study.best_params))   # 최적 kwargs 재구성
    Cls, fixed = BUILDERS[name]
    TUNED[name] = make_factory(Cls, bp, fixed)                       # Stage 4에서 재사용
    s = summarize_cv(run_cv(COLS, TUNED[name]))
    m, sd = s.loc["PR_AUC", "mean"], s.loc["PR_AUC", "std"]
    rows.append({
        "모델": name, "PR_AUC": m, "PR_std": sd, "PR_보수적": m - sd,
        "ROC_AUC": s.loc["ROC_AUC", "mean"], "Brier": s.loc["Brier", "mean"],
        "기본대비Δ": m - DEFAULT_PR.get(name, np.nan),
    })

table3 = pd.DataFrame(rows).sort_values("PR_AUC", ascending=False).reset_index(drop=True)
print("=== Stage 3 튜닝 후 비교 — PR-AUC 내림차순 ===")
print("(바닥선 0.091 / 기본대비Δ = 튜닝 - Stage 2 기본)\n")
print(table3.round(4).to_string(index=False))

for name, study in studies.items():
    print(f"\n[{name}] best params:\n  {study.best_params}")



=== RF 튜닝 (40 trials) ===
  best mean PR-AUC = 0.3396

=== XGBoost 튜닝 (40 trials) ===
  best mean PR-AUC = 0.3247

=== LightGBM 튜닝 (40 trials) ===
  best mean PR-AUC = 0.3233

=== CatBoost 튜닝 (40 trials) ===
  best mean PR-AUC = 0.3401
=== Stage 3 튜닝 후 비교 — PR-AUC 내림차순 ===
(바닥선 0.091 / 기본대비Δ = 튜닝 - Stage 2 기본)

      모델  PR_AUC  PR_std  PR_보수적  ROC_AUC  Brier  기본대비Δ
CatBoost  0.3401  0.1009  0.2392   0.7881 0.0769 0.0510
      RF  0.3396  0.0979  0.2417   0.7881 0.0777 0.0301
 XGBoost  0.3247  0.0826  0.2421   0.7833 0.0914 0.0839
LightGBM  0.3233  0.0797  0.2436   0.7866 0.0796 0.0620

[RF] best params:
  {'class_weight': 'none', 'n_estimators': 288, 'max_depth': 6, 'max_features': 0.38881093442104125, 'min_samples_leaf': 3, 'min_samples_split': 7}

[XGBoost] best params:
  {'n_estimators': 273, 'max_depth': 6, 'learning_rate': 0.011240768803005551, 'subsample': 0.9637281608315128, 'colsample_bytree': 0.7035119926400067, 'min_child_weight': 7, 'reg_lambda': 0.017654048052495083, 'reg

# Stage 4. 앙상블

In [23]:
# %% [Cell E0] 베이스 모델 선택 & OOF 예측 생성 ----------------------------
import numpy as np
import pandas as pd

STAGE4_FS = "S3_정적+캐나다"
COLS = feature_sets[STAGE4_FS]

try:
    BASE = dict(TUNED)
    print("✔ 튜닝본(TUNED) 사용:", list(BASE))
except NameError:
    BASE = dict(MODELS2)
    print("⚠ TUNED 없음 → Stage 2 기본 모델(MODELS2)로 진행. 튜닝 완료 후 재실행 권장.")
    print("  베이스:", list(BASE))


def oof_predictions(base_factories, feature_cols, cv_folds=None):
    """각 베이스 모델의 OOF 확률(검증폴드에서만 예측)을 모은다. 테스트 폴드는 미사용."""
    cv_folds = cv_folds or CV_FOLDS
    idx = df.index[df["_fold"].isin(cv_folds)]
    oof = {n: pd.Series(index=idx, dtype=float) for n in base_factories}
    for vf in cv_folds:
        tr = df.index[df["_fold"].isin([f for f in cv_folds if f != vf])]
        va = df.index[df["_fold"] == vf]
        for n, fac in base_factories.items():
            m = fac()
            m.fit(feat.loc[tr, feature_cols], df.loc[tr, TARGET])
            oof[n].loc[va] = m.predict_proba(feat.loc[va, feature_cols])[:, 1]
    oof_df = pd.DataFrame(oof)
    return oof_df, df.loc[idx, TARGET], df.loc[idx, "_fold"]


oof_df, y_oof, fold_oof = oof_predictions(BASE, COLS)
print(f"OOF 예측 완료: {oof_df.shape[0]}행 × 베이스 {oof_df.shape[1]}개")

✔ 튜닝본(TUNED) 사용: ['RF', 'XGBoost', 'LightGBM', 'CatBoost']
OOF 예측 완료: 12940행 × 베이스 4개


In [24]:
# %% [Cell E1] 단일 vs 단순평균 vs 랭크평균 비교 ---------------------------
y = y_oof.to_numpy()
folds = fold_oof.to_numpy()

scores = {n: oof_df[n] for n in oof_df.columns}  # 단일 모델들
scores["단순평균"] = oof_df.mean(axis=1)
scores["랭크평균"] = oof_df.rank(pct=True).mean(axis=1)  # 스케일 무관, PR-AUC에 적합


def per_fold_prauc(sc):
    s = np.asarray(sc)
    return [evaluate(y[folds == vf], s[folds == vf])["PR_AUC"] for vf in CV_FOLDS]


rows, perfold = [], {}
for name, sc in scores.items():
    per = per_fold_prauc(sc)
    perfold[name] = per
    rows.append(
        {
            "방법": name,
            "PR_AUC": np.mean(per),
            "PR_std": np.std(per),
            "PR_보수적": np.mean(per) - np.std(per),
        }
    )

tableE = (
    pd.DataFrame(rows).sort_values("PR_AUC", ascending=False).reset_index(drop=True)
)
print("=== 단일 vs 앙상블 — PR-AUC 내림차순 ===\n")
print(tableE.round(4).to_string(index=False))
print("\n=== 방법 × 폴드 PR-AUC ===")
print(
    pd.DataFrame(perfold, index=[f"fold{f}" for f in CV_FOLDS]).T.round(3).to_string()
)

# %% [Cell E2] (선택) Optuna 가중평균 + 최종 판단 --------------------------
best_single = max(oof_df.columns, key=lambda n: np.mean(perfold[n]))
best_single_pr = np.mean(perfold[best_single])

try:
    import optuna

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def w_obj(trial):
        w = np.array([trial.suggest_float(f"w_{n}", 0.0, 1.0) for n in oof_df.columns])
        if w.sum() == 0:
            return 0.0
        sc = (oof_df.values * (w / w.sum())).sum(axis=1)
        return float(np.mean(per_fold_prauc(sc)))

    st = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    st.optimize(w_obj, n_trials=300)  # 모델 재학습 없음 → 수초
    w = np.array([st.best_params[f"w_{n}"] for n in oof_df.columns])
    w = w / w.sum()
    wsc = (oof_df.values * w).sum(axis=1)
    wper = per_fold_prauc(wsc)
    print(
        "가중평균 최적 가중치:",
        {n: round(float(x), 3) for n, x in zip(oof_df.columns, w)},
    )
    print(
        f"가중평균 PR-AUC = {np.mean(wper):.4f} ± {np.std(wper):.4f}  (OOF 적합 → 낙관적)"
    )
except ImportError:
    print("optuna 미설치 → 가중평균 생략")
    wper = None

# --- 최종 판단 (간결) ---
best_ens = max(["단순평균", "랭크평균"], key=lambda n: np.mean(perfold[n]))
best_ens_pr = np.mean(perfold[best_ens])
print(f"\n최고 단일: {best_single} (PR-AUC {best_single_pr:.4f})")
print(f"최고 앙상블(누수無): {best_ens} (PR-AUC {best_ens_pr:.4f})")
gain = best_ens_pr - best_single_pr
if gain > 0.01:
    print(f"→ 앙상블 채택 권장: 단일 대비 +{gain:.4f} (분산도 함께 확인)")
else:
    print(f"→ 단일 모델 유지 권장: 앙상블 이득 +{gain:.4f}로 미미 (간결성 우선)")
print("※ 최종 확정은 Stage 5에서 격리 테스트 폴드(fold 0) 1회로.")

=== 단일 vs 앙상블 — PR-AUC 내림차순 ===

      방법  PR_AUC  PR_std  PR_보수적
CatBoost  0.3401  0.1009  0.2392
      RF  0.3396  0.0979  0.2416
    단순평균  0.3376  0.0855  0.2520
    랭크평균  0.3371  0.0837  0.2534
 XGBoost  0.3247  0.0826  0.2421
LightGBM  0.3233  0.0797  0.2436

=== 방법 × 폴드 PR-AUC ===
          fold1  fold2  fold3  fold4
RF        0.502  0.270  0.330  0.257
XGBoost   0.462  0.259  0.318  0.260
LightGBM  0.454  0.268  0.321  0.250
CatBoost  0.502  0.293  0.335  0.230
단순평균      0.477  0.279  0.336  0.258
랭크평균      0.472  0.279  0.341  0.256
가중평균 최적 가중치: {'RF': 0.485, 'XGBoost': 0.045, 'LightGBM': 0.116, 'CatBoost': 0.354}
가중평균 PR-AUC = 0.3461 ± 0.0972  (OOF 적합 → 낙관적)

최고 단일: CatBoost (PR-AUC 0.3401)
최고 앙상블(누수無): 단순평균 (PR-AUC 0.3376)
→ 단일 모델 유지 권장: 앙상블 이득 +-0.0025로 미미 (간결성 우선)
※ 최종 확정은 Stage 5에서 격리 테스트 폴드(fold 0) 1회로.


# Stage 5.

In [ ]:
# %% [Cell F0] 분할 준비 & OOF 재사용 -------------------------------------
import numpy as np
import pandas as pd

STAGE5_FS = "S3_정적+캐나다"
COLS = feature_sets[STAGE5_FS]

cv_idx = df.index[df["_fold"].isin(CV_FOLDS)]
test_idx = df.index[df["_fold"] == 0]
y_cv = df.loc[cv_idx, TARGET]
y_test = df.loc[test_idx, TARGET].to_numpy()

try:
    OOF, Y_OOF = oof_df.copy(), y_oof.copy()  # Stage 4 OOF (임계값 선택용)
except NameError:
    raise SystemExit("oof_df/y_oof 없음 → Stage 4를 먼저 실행하세요.")

print(
    f"CV풀: {len(cv_idx)}행(양성 {int(y_cv.sum())})  |  "
    f"테스트(fold0): {len(test_idx)}행(양성 {int(y_test.sum())})"
)

In [ ]:
# %% [Cell F1] CV풀 전체 재학습 → 테스트 폴드 예측 ------------------------
test_pred = {}
for name, fac in TUNED.items():
    m = fac()
    m.fit(feat.loc[cv_idx, COLS], y_cv)
    test_pred[name] = m.predict_proba(feat.loc[test_idx, COLS])[:, 1]
test_pred = pd.DataFrame(test_pred, index=test_idx)

# 후보별 점수 (테스트 / OOF)
cand_test = {
    "LightGBM_단일": test_pred["LightGBM"].to_numpy(),
    "랭크평균_앙상블": test_pred.rank(pct=True).mean(axis=1).to_numpy(),
}
cand_oof = {
    "LightGBM_단일": OOF["LightGBM"].to_numpy(),
    "랭크평균_앙상블": OOF.rank(pct=True).mean(axis=1).to_numpy(),
}
print("재학습 완료. 후보:", list(cand_test))

# %% [Cell F2] 임계값(OOF) + 테스트 폴드 1회 평가 -------------------------
CV_REF = {"LightGBM_단일": 0.3233, "랭크평균_앙상블": 0.3371}  # 참고: CV 평균 PR-AUC

rows = []
for name in cand_test:
    thr = pick_threshold(Y_OOF.to_numpy(), cand_oof[name])  # 검증(OOF)에서 결정
    ev = evaluate(y_test, cand_test[name], threshold=thr)  # 테스트 1회
    ev["후보"] = name
    ev["CV참고"] = CV_REF.get(name, np.nan)
    rows.append(ev)

final = pd.DataFrame(rows).set_index("후보")
print("=== Stage 5 최종 — 격리 테스트 폴드(fold 0) 평가 ===")
print("(바닥선 PR-AUC ≈ 0.091 / CV참고 = 교차검증 평균과 비교)\n")
print(
    final[
        [
            "PR_AUC",
            "CV참고",
            "ROC_AUC",
            "Brier",
            "Precision",
            "Recall",
            "F1",
            "threshold",
        ]
    ]
    .round(4)
    .to_string()
)
print("\n해석: 테스트 PR-AUC가 CV참고와 비슷하면 일반화가 정직하게 잡힌 것.")
print("      크게 낮으면 과적합/선택편향 의심.")

# %% [Cell F3] SHAP — 변수 기여 서사 (LightGBM 대표) ----------------------
try:
    import shap

    lgbm = TUNED["LightGBM"]()
    lgbm.fit(feat.loc[cv_idx, COLS], y_cv)
    Xs = feat.loc[test_idx, COLS]
    sv = shap.TreeExplainer(lgbm).shap_values(Xs)
    if isinstance(sv, list):
        sv1 = sv[1]
    elif getattr(sv, "ndim", 2) == 3:
        sv1 = sv[:, :, 1]
    else:
        sv1 = sv
    imp = pd.Series(np.abs(sv1).mean(axis=0), index=COLS).sort_values(ascending=False)
    print("=== SHAP 변수 중요도 (평균 |SHAP|, 상위 15) ===")
    print(imp.head(15).round(4).to_string())
    # 그림이 필요하면: shap.summary_plot(sv1, Xs, max_display=15)
except ImportError:
    print("shap 미설치 → pip install shap")